In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, classification_report

# Faz 2'de kaydettiğimiz işlenmiş verileri yükleyelim
# Eğer kaydetmediysen, önceki adımlardaki X_train_final, y_train, X_val_final, y_val değişkenlerini kullanabilirsin.
print("İşlenmiş veriler yükleniyor...")
train_df = pd.read_csv('train_processed.csv').dropna()
val_df = pd.read_csv('val_processed.csv').dropna()

X_train = train_df['address']
y_train = train_df['label']
X_val = val_df['address']
y_val = val_df['label']

print("Veriler yüklendi.")
print(f"Eğitim seti boyutu: {len(X_train)}, Doğrulama seti boyutu: {len(X_val)}")

İşlenmiş veriler yükleniyor...
Veriler yüklendi.
Eğitim seti boyutu: 678586, Doğrulama seti boyutu: 169648


In [2]:
# Model pipeline'ını oluşturuyoruz
# Bu parametreler başlangıç için çok iyidir ve seni öne çıkarabilir.
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='char_wb',      # Hem kelime hem karakter seviyesinde çalışır. Yazım hatalarına dayanıklıdır.
        ngram_range=(3, 5),      # 3'lü, 4'lü ve 5'li karakter gruplarını dikkate alır. "istan" ve "istanbul" benzerliğini yakalar.
        max_features=20000,      # En önemli 100,000 özelliği (karakter n-gram'ı) kullanır.
        sublinear_tf=True        # Kelime frekanslarının etkisini logaritmik olarak yumuşatır.
    )),
    ('clf', LogisticRegression(
        solver='saga',           # Büyük veri setleri için optimize edilmiş bir çözücü.
        C=1.0,                   # Regularizasyon parametresi. Başlangıç için 1.0 iyidir.
        random_state=42,         # Sonuçların tekrarlanabilir olması için.
        n_jobs=-1                # Mevcut tüm CPU çekirdeklerini kullanır, eğitimi hızlandırır.
    ))
])

print("Model pipeline'ı oluşturuldu.")
print(pipeline)

Model pipeline'ı oluşturuldu.
Pipeline(steps=[('tfidf',
                 TfidfVectorizer(analyzer='char_wb', max_features=20000,
                                 ngram_range=(3, 5), sublinear_tf=True)),
                ('clf',
                 LogisticRegression(n_jobs=-1, random_state=42,
                                    solver='saga'))])


In [3]:
# Modeli eğitme (Bu işlem veri setinin büyüklüğüne ve bilgisayarınızın gücüne bağlı olarak zaman alabilir)
print("\nModel eğitimi başlıyor... Bu işlem biraz sürebilir.")
pipeline.fit(X_train, y_train)
print("Model eğitimi tamamlandı.")

# Doğrulama seti üzerinde tahmin yapma
print("\nDoğrulama seti üzerinde tahminler yapılıyor...")
y_pred = pipeline.predict(X_val)
print("Tahminler yapıldı.")

# Performansı hesaplama
# 'macro' F1 skoru, her sınıfın F1 skorunun ağırlıksız ortalamasını alır.
# Dengesiz veri setleri için daha dürüst bir metriktir.
macro_f1 = f1_score(y_val, y_pred, average='macro')
# 'weighted' F1 skoru, her sınıfın F1 skorunu o sınıfın örnek sayısıyla ağırlıklandırarak ortalama alır.
weighted_f1 = f1_score(y_val, y_pred, average='weighted')

print(f"\nMacro F1 Score: {macro_f1:.4f}")
print(f"Weighted F1 Score: {weighted_f1:.4f}")

# Detaylı sınıflandırma raporu
# Bu rapor, her bir sınıf için Precision, Recall ve F1-Score değerlerini gösterir.
# Ancak 10,390 sınıf olduğu için ekrana yazdırmak çok uzun sürebilir.
# Sadece birkaç sınıfın performansına bakmak veya raporu bir dosyaya yazmak daha mantıklı olabilir.
print("\nSınıflandırma Raporu (İlk 10 sınıf için örnek):")
# Tüm sınıfları göstermek yerine, rapordaki etiket sayısını sınırlayalım.
unique_labels_in_val = sorted(y_val.unique())
target_names_subset = [str(label) for label in unique_labels_in_val[:10]]
print(classification_report(y_val, y_pred, target_names=target_names_subset, labels=unique_labels_in_val[:10]))


Model eğitimi başlıyor... Bu işlem biraz sürebilir.


MemoryError: Unable to allocate 52.5 GiB for an array with shape (678586, 10390) and data type float64